In [15]:
import sys

print("Version Python :", sys.version)

Version Python : 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


Installation du pakage ffmpeg

In [16]:
!apt-get update -qq
!apt-get install -y ffmpeg

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.


In [17]:
!ffmpeg -version

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --enable-

In [18]:
import torch

print("PyTorch :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("Exécution sur CPU")

PyTorch : 2.14.1+cpu
GPU disponible : False
Exécution sur CPU


In [19]:
!pip install -q faster-whisper

In [20]:
!pip install -q nvidia-cublas-cu12

In [21]:
import faster_whisper

print("faster-whisper installé :", faster_whisper.__version__)

faster-whisper installé : 1.2.1


In [22]:
import subprocess
from pathlib import Path

# Afficher les fichiers disponibles
files = list(Path("/content").glob("*"))

for file in files:
    print(file.name)

# À adapter avec le nom exact de ta vidéo
video_path = "/content/inventa_pub (1).mp4"
audio_path = "/content/audio_test.wav"

command = [
    "ffmpeg", "-y",
    "-i", video_path,
    "-vn",
    "-ac", "1",
    "-ar", "16000",
    "-c:a", "pcm_s16le",
    audio_path
]

result = subprocess.run(command, capture_output=True, text=True)

if result.returncode == 0:
    print("\nExtraction réussie :", audio_path)
    print("Taille :", Path(audio_path).stat().st_size, "octets")
else:
    print("Erreur :", result.stderr[-2000:])

.config
ai-video-dubbing
sample_data
Erreur : ht (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libz

In [25]:
from pathlib import Path

# Chercher tous les fichiers vidéo MP4 dans /content
mp4_files = list(Path('/content').glob('*.mp4'))

if mp4_files:
    print("Fichiers vidéo MP4 trouvés :")
    for f in mp4_files:
        print(f"- {f.name}")
else:
    print("Aucun fichier .mp4 n'a été trouvé dans /content. Veuillez téléverser votre vidéo.")

Aucun fichier .mp4 n'a été trouvé dans /content. Veuillez téléverser votre vidéo.


### Étape 1 : Téléversement de votre vidéo `inventa_pub (1).mp4`
Exécutez la cellule suivante pour téléverser votre fichier vidéo directement dans l'environnement `/content`.

In [26]:
from google.colab import files
import os

print("Veuillez sélectionner et téléverser votre fichier vidéo (idéalement nommé 'inventa_pub (1).mp4') :")
uploaded = files.upload()

# Renommer ou vérifier le fichier téléversé pour correspondre à notre pipeline
for filename in uploaded.keys():
    if "inventa_pub" in filename:
        os.rename(filename, "/content/inventa_pub (1).mp4")
        print("\nFichier prêt et correctement positionné sous : /content/inventa_pub (1).mp4")

Veuillez sélectionner et téléverser votre fichier vidéo (idéalement nommé 'inventa_pub (1).mp4') :


Saving inventa_pub.mp4 to inventa_pub.mp4

Fichier prêt et correctement positionné sous : /content/inventa_pub (1).mp4


Extraction du de l'audio en text

In [27]:
import os
import sys
import ctypes
import importlib
import subprocess
from pathlib import Path
import av
import torch

# --- 1. EXTRACTION AUDIO VIA FFMPEG ---
video_path = "/content/inventa_pub (1).mp4"
audio_path = "/content/audio_test.wav"

print("=== ÉTAPE 1 : EXTRACTION AUDIO ===")
if os.path.exists(video_path):
    command = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "pcm_s16le",
        audio_path
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode == 0:
        print("Extraction réussie :", audio_path)
        print("Taille de l'audio :", Path(audio_path).stat().st_size, "octets")
    else:
        print("Erreur FFmpeg :", result.stderr[-1000:])
else:
    print(f"Erreur : La vidéo {video_path} n'existe pas. Veuillez la téléverser d'abord.")

# --- 2. TRANSCRIPTION AVEC FASTER-WHISPER ---
print("\n=== ÉTAPE 2 : TRANSCRIPTION DE L'AUDIO ===")
# Recharger proprement le module pour éliminer les anciennes fonctions de patch
importlib.reload(av)

# Patch av.open de manière propre
av._original_open = av.open
def patched_open(*args, **kwargs):
    kwargs.pop('metadata_errors', None)
    return av._original_open(*args, **kwargs)
av.open = patched_open

# Déterminer le périphérique de calcul disponible
if torch.cuda.is_available():
    device = "cuda"
    compute_type = "float16"
    print("Utilisation du GPU (CUDA).")
    python_version = f"python{sys.version_info.major}.{sys.version_info.minor}"
    cublas_dir = f"/usr/local/lib/{python_version}/dist-packages/nvidia/cublas/lib"
    libcublas_path = os.path.join(cublas_dir, "libcublas.so.12")
    libcublaslt_path = os.path.join(cublas_dir, "libcublasLt.so.12")
    if os.path.exists(libcublaslt_path):
        try:
            ctypes.CDLL(libcublaslt_path, mode=ctypes.RTLD_GLOBAL)
        except Exception as e: pass
    if os.path.exists(libcublas_path):
        try:
            ctypes.CDLL(libcublas_path, mode=ctypes.RTLD_GLOBAL)
        except Exception as e: pass
else:
    device = "cpu"
    compute_type = "int8"
    print("Utilisation du CPU.")

from faster_whisper import WhisperModel

if os.path.exists(audio_path):
    model = WhisperModel("base", device=device, compute_type=compute_type)
    segments, info = model.transcribe(audio_path, language="fr", beam_size=5)

    print("\n--- RÉSULTATS DE LA TRANSCRIPTION ---")
    print("Langue détectée :", info.language)
    print("Durée audio :", round(info.duration, 2), "secondes\n")

    for segment in segments:
        print(f"[{segment.start:.2f}s → {segment.end:.2f}s] {segment.text}")
else:
    print("Impossible de démarrer la transcription sans le fichier audio.")

=== ÉTAPE 1 : EXTRACTION AUDIO ===
Extraction réussie : /content/audio_test.wav
Taille de l'audio : 1573090 octets

=== ÉTAPE 2 : TRANSCRIPTION DE L'AUDIO ===
Utilisation du CPU.

--- RÉSULTATS DE LA TRANSCRIPTION ---
Langue détectée : fr
Durée audio : 49.16 secondes

[0.00s → 4.10s]  ...
[4.10s → 8.90s]  Étant jeux. Avec inventat, prenez le contrôle de votre activité.
[8.90s → 14.20s]  Suivez chaque bijou avec précision, poids, prix du gramme et mouvement de stock.
[14.20s → 17.90s]  Gardez une vision claire et actualisée de votre inventaire.
[17.90s → 23.40s]  Côtéventes, simplifiez vos opérations. Enquaissement rapide, gestion des accounts,
[23.40s → 25.80s]  suivis des paiements et des soldes clients.
[25.80s → 32.00s]  Générer des reçus professionnels avec QR code de vérification et partager les instantanément.
[32.00s → 37.00s]  Et surtout accéder à vos données à tout moment, depuis votre boutique ou à distance.
[37.00s → 40.00s]  Moins d'erreur, plus de contrôle.
[40.00s → 46.20

### Étape 3 : Traduction du texte transcrit (Français → Anglais)
Nous importons et appliquons le module de traduction présent dans le dépôt cloné `ai-video-dubbing` sur notre texte transcrit.

In [30]:
import sys
import os

# Ajouter le dossier src au chemin de recherche
sys.path.append('/content/ai-video-dubbing')

try:
    # Importer la classe HuggingFaceTranslator depuis le module de traduction
    from src.translation.text_translation import HuggingFaceTranslator

    # Exemple de texte transcrit à traduire
    text_to_translate = "Avec inventat, prenez le contrôle de votre activité. Suivez chaque bijou avec précision."

    print("=== TRADUCTION AVEC LE MODÈLE HUGGING FACE ===")
    print("Texte d'origine (FR) :", text_to_translate)

    # Initialisation du traducteur
    translator = HuggingFaceTranslator()

    # Méthode d'exécution (utilisation de la méthode de traduction interne ou traduction directe)
    # Si le traducteur Hugging Face a une méthode translate
    if hasattr(translator, 'translate'):
        translated_text = translator.translate(text_to_translate)
    else:
        # Fallback au cas où l'implémentation diffère
        inputs = translator.tokenizer(text_to_translate, return_tensors="pt").to(translator.device)
        outputs = translator.model.generate(**inputs, **translator.generation_params)
        translated_text = translator.tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\nTexte traduit (EN) :", translated_text)
except Exception as e:
    print("Une erreur est survenue lors du processus de traduction :", e)

=== TRADUCTION AVEC LE MODÈLE HUGGING FACE ===
Texte d'origine (FR) : Avec inventat, prenez le contrôle de votre activité. Suivez chaque bijou avec précision.


config.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

source.spm: reconstructing file:   0%|          |  0.00B /  820kB            

source.spm: downloading bytes:           |  0.00B            

target.spm: reconstructing file:   0%|          |  0.00B /  802kB            

... (6 outputs left out)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Texte traduit (EN) : With inventiveness, take control of your business. Follow each jewel with precision.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q --upgrade av

In [1]:
!git clone https://github.com/Abdouaziz2/ai-video-dubbing.git
%cd ai-video-dubbing

print("\n===== STRUCTURE DU PROJET =====")
!find . -maxdepth 3 -type f | sort

print("\n===== DÉPENDANCES =====")
!ls -la requirements*

Cloning into 'ai-video-dubbing'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 38 (delta 1), reused 38 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 30.19 KiB | 2.74 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/ai-video-dubbing

===== STRUCTURE DU PROJET =====
./data/input/.gitkeep
./data/intermediate/.gitkeep
./data/output/.gitkeep
./docs/data_contract.md
./docs/progress.md
./docs/translation_model_choice.md
./.git/config
./.git/description
./.git/HEAD
./.git/hooks/applypatch-msg.sample
./.git/hooks/commit-msg.sample
./.git/hooks/fsmonitor-watchman.sample
./.git/hooks/post-update.sample
./.git/hooks/pre-applypatch.sample
./.git/hooks/pre-commit.sample
./.git/hooks/pre-merge-commit.sample
./.git/hooks/prepare-commit-msg.sample
./.git/hooks/pre-push.sample
./.git/hooks/pre-rebase.sample
./.git/hooks/pre-receive.sample
./.git/hooks/push-to-checkout.sam

### Lecture des fichiers de configuration et dépendances

Nous allons afficher le contenu du fichier `requirements.txt` pour identifier les paquets requis pour le projet de doublage vidéo.

In [ ]:
import os
from pathlib import Path

repo_path = Path('/content/ai-video-dubbing')
if not repo_path.exists():
    print("Dépôt absent. Clonage en cours...")
    !git clone https://github.com/Abdouaziz2/ai-video-dubbing.git /content/ai-video-dubbing

req_path = repo_path / 'requirements.txt'
if req_path.exists():
    print(f"\nFichier trouvé à : {req_path}\n--- Contenu ---")
    print(req_path.read_text(encoding='utf-8'))
else:
    print("Fichier requirements.txt introuvable après tentative de clonage.")

### Exploration du code source de traduction

Regardons le code contenu dans `src/translation/text_translation.py` pour comprendre comment sont gérées les traductions.

In [ ]:
import os
from pathlib import Path

repo_path = Path('/content/ai-video-dubbing')
if not repo_path.exists():
    !git clone https://github.com/Abdouaziz2/ai-video-dubbing.git /content/ai-video-dubbing

trans_path = repo_path / 'src/translation/text_translation.py'
if trans_path.exists():
    print(f"\nFichier trouvé à : {trans_path}\n--- Contenu ---")
    print(trans_path.read_text(encoding='utf-8'))
else:
    print("Fichier text_translation.py introuvable après tentative de clonage.")

In [2]:
from pathlib import Path

contract = Path("/content/ai-video-dubbing/docs/data_contract.md")

if contract.exists():
    print(contract.read_text(encoding="utf-8"))
else:
    print("Fichier introuvable :", contract)

# Contrat de données entre les modules

Tous les fichiers sont en JSON UTF-8, avec une clé racine `segments` (liste ordonnée par `start`).
**Règle commune :** `id`, `start` et `end` ne sont jamais modifiés d'une étape à l'autre.

## 1. Transcription → Traduction (produit par Abdou Aziz)

```json
{"segments": [{"id": 1, "start": 0.0, "end": 3.5, "text": "Bonjour à tous."}]}
```

| Champ | Type | Contrainte |
|---|---|---|
| `id` | entier | unique |
| `start`, `end` | nombre (secondes) | `0 <= start < end` |
| `text` | texte | non vide |

Les segments doivent être triés par `start`. Les champs supplémentaires (ex. infos Whisper)
sont acceptés mais ne sont pas transmis à l'étape suivante.
Validation : `src/translation/translation_utils.py::validate_transcription_segments`.

## 2. Traduction → Voix (produit par Massamba)

```json
{"segments": [{
  "id": 17, "start": 40.0, "end": 50.0,
  "original_text": "Afin de conserver la cohérence...",
  "translated_text": "In order to maintain consist

In [11]:
import pypdf
from pathlib import Path

def read_pdf(file_path):
    try:
        reader = pypdf.PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text
    except Exception as e:
        return f"Erreur : {str(e)}"

repartition_path = Path("/content/Repartition_Projet_4_AI_Video_Dubbing.pdf")

print("=== RELECTURE COMPLÈTE DE LA RÉPARTITION ===")
if repartition_path.exists():
    print(read_pdf(repartition_path))
else:
    print("Fichier de répartition introuvable.")

=== RELECTURE COMPLÈTE DE LA RÉPARTITION ===
Projet 4 — AI Video Dubbing | Répartition de l’équipe Page 1
RÉPARTITION OFFICIELLE DES
RESPONSABILITÉS
Projet 4 — AI Video Dubbing | Équipe de 3 membres
Objectif du projet. Développer une chaîne de doublage vidéo du français vers l’anglais : transcription horodatée,
traduction, génération vocale avec une voix autorisée, synchronisation et assemblage vidéo.
Direction linguistique retenue : français → anglais. Cette direction doit être confirmée avec l’encadreur si le sujet initial
impose une autre direction.
1. Répartition des responsabilités
Membre / rôle Responsabilités principales Fichiers indicatifs
Massamba Fall Séne
NLP & intégration
technique
 Étudier et comparer les modèles de traduction français →
anglais.
 Développer le module de traduction des segments.
 Préserver les identifiants et timestamps.
 Adapter les formulations à l’oral sans changer le sens.
 Définir et documenter le contrat JSON commun.
 Orchestrer le pipeline et 

In [12]:
import os
from pathlib import Path

# Création des répertoires de packages pour les modules d'Abdou Aziz
repo_root = Path("/content/ai-video-dubbing")
extraction_dir = repo_root / "src" / "extraction"
transcription_dir = repo_root / "src" / "transcription"

extraction_dir.mkdir(parents=True, exist_ok=True)
transcription_dir.mkdir(parents=True, exist_ok=True)

# Création des fichiers d'initialisation de package vides s'ils n'existent pas
(extraction_dir / "__init__").with_suffix(".py").touch(exist_ok=True)
(transcription_dir / "__init__").with_suffix(".py").touch(exist_ok=True)

print("Dossiers créés avec succès.")

Dossiers créés avec succès.


In [13]:
%%writefile /content/ai-video-dubbing/src/extraction/audio_extraction.py
import subprocess
from pathlib import Path

def extract_audio_from_video(video_path: str, output_audio_path: str) -> bool:
    """
    Extrait l'audio d'une vidéo au format WAV PCM 16-bit, mono, 16kHz.

    Args:
        video_path: Chemin d'accès du fichier vidéo d'entrée.
        output_audio_path: Chemin d'accès pour enregistrer le fichier audio WAV.

    Returns:
        bool: True si l'extraction a réussi, False sinon.
    """
    video_file = Path(video_path)
    output_file = Path(output_audio_path)

    if not video_file.exists():
        print(f"[Erreur] Le fichier vidéo {video_path} n'existe pas.")
        return False

    output_file.parent.mkdir(parents=True, exist_ok=True)

    command = [
        "ffmpeg", "-y",
        "-i", str(video_file),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "pcm_s16le",
        str(output_file)
    ]

    print(f"Extraction de l'audio depuis {video_file.name}...")
    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode == 0:
        print(f"[Succès] Audio extrait avec succès et sauvegardé dans : {output_file.name}")
        return True
    else:
        print(f"[Erreur] Échec de l'extraction FFmpeg : {result.stderr}")
        return False

Writing /content/ai-video-dubbing/src/extraction/audio_extraction.py


In [14]:
%%writefile /content/ai-video-dubbing/src/transcription/whisper_transcription.py
import os
import sys
import ctypes
import json
import importlib
import av
import torch
from pathlib import Path

def patch_pyav_metadata_issue():
    """
    Monkey-patch de la méthode av.open pour supprimer l'argument metadata_errors
    non supporté dans certaines versions de PyAV.
    """
    if not hasattr(av, '_original_open'):
        av._original_open = av.open
        def patched_open(*args, **kwargs):
            kwargs.pop('metadata_errors', None)
            return av._original_open(*args, **kwargs)
        av.open = patched_open

def preload_cuda_dependencies():
    """
    Précharge manuellement les bibliothèques CUDA nécessaires pour éviter les soucis de chemins
    lorsque les modèles s'exécutent en mode GPU.
    """
    if torch.cuda.is_available():
        python_version = f"python{sys.version_info.major}.{sys.version_info.minor}"
        cublas_dir = f"/usr/local/lib/{python_version}/dist-packages/nvidia/cublas/lib"

        libcublaslt_path = os.path.join(cublas_dir, "libcublasLt.so.12")
        libcublas_path = os.path.join(cublas_dir, "libcublas.so.12")

        if os.path.exists(libcublaslt_path):
            try:
                ctypes.CDLL(libcublaslt_path, mode=ctypes.RTLD_GLOBAL)
            except Exception:
                pass
        if os.path.exists(libcublas_path):
            try:
                ctypes.CDLL(libcublas_path, mode=ctypes.RTLD_GLOBAL)
            except Exception:
                pass

def transcribe_audio(audio_path: str, output_json_path: str, model_size: str = "base", language: str = "fr") -> dict:
    """
    Transcrit l'audio d'entrée avec Faster-Whisper, génère un dictionnaire au format requis
    par le contrat de données, et l'exporte en fichier JSON UTF-8.
    """
    audio_file = Path(audio_path)
    if not audio_file.exists():
        raise FileNotFoundError(f"Fichier audio introuvable : {audio_path}")

    patch_pyav_metadata_issue()
    preload_cuda_dependencies()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"

    print(f"Initialisation du modèle Faster-Whisper ({model_size}) sur périphérique : {device}...")

    from faster_whisper import WhisperModel
    model = WhisperModel(model_size, device=device, compute_type=compute_type)

    print("Début de la transcription...")
    segments, info = model.transcribe(str(audio_file), language=language, beam_size=5)

    output_segments = []
    for i, segment in enumerate(segments, start=1):
        # Respect rigoureux du contrat JSON spécifié dans docs/data_contract.md
        output_segments.append({
            "id": i,
            "start": round(segment.start, 2),
            "end": round(segment.end, 2),
            "text": segment.text.strip()
        })

    data_payload = {
        "segments": output_segments
    }

    output_path = Path(output_json_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data_payload, f, ensure_ascii=False, indent=2)

    print(f"[Succès] Transcription terminée. Données exportées dans {output_path.name}")
    return data_payload

Writing /content/ai-video-dubbing/src/transcription/whisper_transcription.py


In [16]:
!pip install -q av

import sys
# Ajouter temporairement le dépôt au chemin d'import
if "/content/ai-video-dubbing" not in sys.path:
    sys.path.append("/content/ai-video-dubbing")

from src.extraction.audio_extraction import extract_audio_from_video
from src.transcription.whisper_transcription import transcribe_audio

# Exécution bout-en-bout locale de validation des modules
video_input = "/content/inventa_pub (1).mp4"
audio_tmp = "/content/ai-video-dubbing/data/intermediate/audio_extracted.wav"
json_output = "/content/ai-video-dubbing/data/intermediate/transcription_fr.json"

# Étape 1 : Extraction
if extract_audio_from_video(video_input, audio_tmp):
    # Étape 2 : Transcription
    resultat_json = transcribe_audio(audio_tmp, json_output, model_size="base", language="fr")
    print("\n--- Aperçu du JSON généré (Conforme au contrat) ---")
    import json
    print(json.dumps(resultat_json, indent=2, ensure_ascii=False)[:1000])

[Erreur] Le fichier vidéo /content/inventa_pub (1).mp4 n'existe pas.


In [18]:
# Configuration des informations d'authentification Git
# Remplacer les valeurs ci-dessous par vos propres informations si nécessaire
!git config --global user.name "abdouaziz2"
!git config --global user.email "abdouazizseck2015@gmail.com"

# Création ou passage sur votre branche 'abdou'
!git checkout -b abdou 2>/dev/null || git checkout abdou

# Ajout des nouveaux fichiers de modules
!git add src/extraction/audio_extraction.py src/transcription/whisper_transcription.py

# Commit propre et concis rédigé de façon humaine
!git commit -m "Ajout des modules d'extraction audio et de transcription ASR avec Faster-Whisper"

# Note: Pour pousser sur votre dépôt distant, l'authentification GitHub (token) peut être requise.
# Si nécessaire, décommentez et configurez la ligne suivante :
# !git push https://<VOTRE_TOKEN_GITHUB>@github.com/Abdouaziz2/ai-video-dubbing.git abdou
print("Code commité localement avec succès sur la branche abdou.")

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
Code commité localement avec succès sur la branche abdou.
